In [20]:
%pip install langchain langchain[mistralai] redis redisvl langgraph-checkpoint-redis langmem

   ---------------------------------------- 0.0/67.1 kB ? eta -:--:--
   ---------------------------------------- 67.1/67.1 kB 1.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/47.5 kB ? eta -:--:--
   ---------------------------------------- 47.5/47.5 kB 2.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/455.2 kB ? eta -:--:--
   ---------------------------------------- 455.2/455.2 kB 9.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="mistralai/ministral-3-3b",
    model_provider="mistralai",
    base_url ="http://localhost:1234/v1",
    api_key = "not needed"
)
response = model.invoke('what is the capital of france?')
print(response.content)


c:\Users\stony\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The capital of France is **Paris**.


In [9]:
import os

import redis

# Use the environment variable if set, otherwise default to localhost
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
redis_client = redis.Redis(
        host=os.getenv("REDIS_HOST", "localhost"),
        port=int(os.getenv("REDIS_PORT", 6379)),
        decode_responses=True
    )
# redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()

True

In [ ]:
import ulid

from datetime import datetime
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field


class MemoryType(str, Enum):
    """
    Defines the type of long-term memory for categorization and retrieval.

    EPISODIC: Personal experiences and user-specific preferences
              (e.g., "User prefers Delta airlines", "User visited Paris last year")

    SEMANTIC: General domain knowledge and facts
              (e.g., "Singapore requires passport", "Tokyo has excellent public transit")

    The type of a long-term memory.

    EPISODIC: User specific experiences and preferences

    SEMANTIC: General knowledge on top of the user's preferences and LLM's
    training data.
    """

    EPISODIC = "episodic"
    SEMANTIC = "semantic"


class Memory(BaseModel):
    """Represents a single long-term memory."""

    content: str
    memory_type: MemoryType
    metadata: str


class Memories(BaseModel):
    """
    A list of memories extracted from a conversation by an LLM.

    NOTE: OpenAI's structured output requires us to wrap the list in an object.
    """

    memories: List[Memory]


class StoredMemory(Memory):
    """A stored long-term memory"""

    id: str  # The redis key
    memory_id: ulid.ULID = Field(default_factory=lambda: ulid.ULID())
    created_at: datetime = Field(default_factory=datetime.now)
    user_id: Optional[str] = None
    thread_id: Optional[str] = None
    memory_type: Optional[MemoryType] = None

In [26]:
from redisvl.index import SearchIndex
from redisvl.schema.schema import IndexSchema


# Define the schema for our vector search index
# This creates the structure for storing and querying memories
memory_schema = IndexSchema.from_dict({
        "index": {
            "name": "agent_memories",  # Index name for identification
            "prefix": "memory",       # Redis key prefix (memory:1, memory:2, etc.)
            "key_separator": ":",
            "storage_type": "json",
        },
        "fields": [
            {"name": "content", "type": "text"},
            {"name": "memory_type", "type": "tag"},
            {"name": "metadata", "type": "text"},
            {"name": "created_at", "type": "text"},
            {"name": "user_id", "type": "tag"},
            {"name": "memory_id", "type": "tag"},
            {
                "name": "embedding",
                "type": "vector",
                "attrs": {
                    "algorithm": "flat",
                    "dims": 768,  # OpenAI embedding dimension
                    "distance_metric": "cosine",
                    "datatype": "float32",
                },
            },
        ],
    }
)

In [27]:
try:
    long_term_memory_index = SearchIndex(
        schema=memory_schema,
        redis_client=redis_client,
        validate_on_load=True
    )
    long_term_memory_index.create(overwrite=True)
    print("Long-term memory index ready")
except Exception as e:
    print(f"Error creating index: {e}")

Long-term memory index ready


In [29]:


!rvl index info -i agent_memories





Index Information:
╭────────────────┬────────────────┬────────────────┬────────────────┬────────────────╮
│ Index Name     │ Storage Type   │ Prefixes       │ Index Options  │ Indexing       │
├────────────────┼────────────────┼────────────────┼────────────────┼────────────────┤
| agent_memories | JSON           | ['memory']     | []             | 0              |
╰────────────────┴────────────────┴────────────────┴────────────────┴────────────────╯
Index Fields:
╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────╮
│ Name            │ Attribute       │ Type            │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │
├─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼───

In [30]:

from redisvl.utils.vectorize.text.openai import OpenAITextVectorizer

openai_embed = OpenAITextVectorizer(model="text-embedding-embeddinggemma-300m-qat",
                                     api_config={
        "api_key": "lm-studio",
        "base_url": "http://localhost:1234/v1"
    })
openai_embed.embed("hello world")



[0.06518006324768066,
 0.03697087615728378,
 -0.02857222408056259,
 0.060351528227329254,
 -0.018400846049189568,
 -0.06646353006362915,
 -0.009071076288819313,
 0.030498310923576355,
 0.04036334902048111,
 0.007408133242279291,
 -0.025620045140385628,
 -0.06968115270137787,
 -0.007603147532790899,
 0.030059263110160828,
 -0.032373156398534775,
 0.012694835662841797,
 0.0094306580722332,
 0.01014748215675354,
 0.015632448717951775,
 -0.03843635693192482,
 0.014860734343528748,
 0.014885924756526947,
 0.033500321209430695,
 -0.010285691358149052,
 0.05420108884572983,
 0.010422068648040295,
 0.05397753790020943,
 0.026062212884426117,
 -0.0015436774119734764,
 -0.03011947125196457,
 0.03109610825777054,
 -0.03569457679986954,
 -0.02928142622113228,
 -0.03112734854221344,
 0.017405914142727852,
 -0.030641287565231323,
 0.024794531986117363,
 0.022319091483950615,
 -0.0015010893112048507,
 -0.32229161262512207,
 0.07345181703567505,
 0.012572244741022587,
 -0.020725011825561523,
 0.004576

In [31]:
import logging

# Set up a logger
logger = logging.getLogger(__name__)

In [32]:
'''1. Check for similar memories

First, we'll write a utility function to check if a memory similar to a given memory already exists in the index.

This function checks for duplicate memories in Redis by:

    Converting the input content into a vector embedding
    Creating filters for user_id and memory_type
    Using vector similarity search with a vector range query to find any existing + similar memories
    Returning True if a similar memory exists, False otherwise

This helps prevent storing redundant information in the agent's memory.'''

from redisvl.query import VectorRangeQuery
from redisvl.query.filter import Tag


# If we have any memories that aren't associated with a user, we'll use this ID.
SYSTEM_USER_ID = "system"


def similar_memory_exists(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
) -> bool:
    """Check if a similar long-term memory already exists in Redis."""
    content_embedding = openai_embed.embed(content)

    filters = (Tag("user_id") == user_id) & (Tag("memory_type") == memory_type)

    if thread_id:
        filters = filters & (Tag("thread_id") == thread_id)

    # Search for similar memories
    vector_query = VectorRangeQuery(
        vector=content_embedding,
        num_results=1,
        vector_field_name="embedding",
        filter_expression=filters,
        distance_threshold=distance_threshold,
        return_fields=["id"],
    )
    results = long_term_memory_index.query(vector_query)
    logger.debug(f"Similar memory search results: {results}")

    if results:
        logger.debug(
            f"{len(results)} similar {'memory' if results.count == 1 else 'memories'} found. First: "
            f"{results[0]['id']}. Skipping storage."
        )
        return True

    return False

In [33]:
from datetime import datetime
from typing import List, Optional, Union

import ulid


def store_memory(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    metadata: Optional[str] = None,
):
    """Store a long-term memory in Redis with deduplication.

        This function:
        1. Checks for similar existing memories to avoid duplicates
        2. Generates vector embeddings for semantic search
        3. Stores the memory with metadata for retrieval
        """
    if metadata is None:
        metadata = "{}"

    logger.info(f"Preparing to store memory: {content}")

    if similar_memory_exists(content, memory_type, user_id, thread_id):
        logger.info("Similar memory found, skipping storage")
        return

    embedding = openai_embed.embed(content)

    memory_data = {
        "user_id": user_id or SYSTEM_USER_ID,
        "content": content,
        "memory_type": memory_type.value,
        "metadata": metadata,
        "created_at": datetime.now().isoformat(),
        "embedding": embedding,
        "memory_id": str(ulid.ULID()),
        "thread_id": thread_id,
    }

    try:
        long_term_memory_index.load([memory_data])
    except Exception as e:
        logger.error(f"Error storing memory: {e}")
        return

    logger.info(f"Stored {memory_type} memory: {content}")

In [34]:
def retrieve_memories(
    query: str,
    memory_type: Union[Optional[MemoryType], List[MemoryType]] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
    limit: int = 5,
) -> List[StoredMemory]:
    """Retrieve relevant memories from Redis using vector similarity search.

    """
    # Create vector query using query embedding
    logger.debug(f"Retrieving memories for query: {query}")
    vector_query = VectorRangeQuery(
        vector=openai_embed.embed(query),
        return_fields=[
            "content",
            "memory_type", 
            "metadata",
            "created_at",
            "memory_id",
            "thread_id",
            "user_id",
        ],
        num_results=limit,
        vector_field_name="embedding",
        dialect=2,
        distance_threshold=distance_threshold,
    )

    # Build filter conditions
    base_filters = [f"@user_id:{{{user_id or SYSTEM_USER_ID}}}"]

    if memory_type:
        if isinstance(memory_type, list):
            base_filters.append(f"@memory_type:{{{'|'.join(memory_type)}}}")
        else:
            base_filters.append(f"@memory_type:{{{memory_type.value}}}")

    if thread_id:
        base_filters.append(f"@thread_id:{{{thread_id}}}")

    vector_query.set_filter(" ".join(base_filters))

    # Execute vector similarity search
    results = long_term_memory_index.query(vector_query)

    # Parse results into StoredMemory objects
    memories = []
    for doc in results:
        try:
            memory = StoredMemory(
                id=doc["id"],
                memory_id=doc["memory_id"],
                user_id=doc["user_id"],
                thread_id=doc.get("thread_id", None),
                memory_type=MemoryType(doc["memory_type"]),
                content=doc["content"],
                created_at=doc["created_at"],
                metadata=doc["metadata"],
            )
            memories.append(memory)
        except Exception as e:
            logger.error(f"Error parsing memory: {e}")
            continue
    return memories

In [35]:
'''Define Agent Tools

Now that we have our storage functions defined, we can create the tools that will enable our agent to interact with the memory system. These tools will be used by the LLM to manage memories during conversations.

Let's start with the Store Memory Tool:
Store Memory Tool

This tool enables the agent to save important information as long-term memories in Redis. It's particularly useful for capturing:

    User preferences and habits
    Personal experiences and anecdotes
    Important facts and knowledge shared during conversations

The tool accepts the following parameters:

    content: The actual memory content to store (e.g., "User prefers window seats on flights")
    memory_type: The type of memory (e.g., MemoryType.EPISODIC for personal experiences, MemoryType.SEMANTIC for general knowledge)
    metadata: Optional dictionary for additional context (e.g., timestamps, source, confidence)
    config: Optional configuration for user/thread context (automatically handled by the agent)

When called, the tool:

    Validates the input parameters
    Stores the memory in Redis with proper indexing
    Returns a success message with the stored content
    Handles errors gracefully with informative messages

This tool is designed to be used by the LLM to build a persistent memory of the user's preferences and experiences, enabling more personalized and context-aware '''

from typing import Dict, Optional

from langchain_core.tools import tool
from langchain_core.runnables.config import RunnableConfig


@tool
def store_memory_tool(
    content: str,
    memory_type: MemoryType,
    metadata: Optional[Dict[str, str]] = None,
    config: Optional[RunnableConfig] = None,
) -> str:
    """
    Store a long-term memory in the system.

    Use this tool to save important information about user preferences,
    experiences, or general knowledge that might be useful in future
    interactions.
    """
    config = config or RunnableConfig()
    user_id = config.get("user_id", SYSTEM_USER_ID)
    thread_id = config.get("thread_id")

    try:
        # Store in long-term memory
        store_memory(
            content=content,
            memory_type=memory_type,
            user_id=user_id,
            thread_id=thread_id,
            metadata=str(metadata) if metadata else None,
        )

        return f"Successfully stored {memory_type} memory: {content}"
    except Exception as e:
        return f"Error storing memory: {str(e)}"

In [36]:

store_memory_tool.invoke({"content": "I like flying on Delta when possible", "memory_type": "episodic"})



'Successfully stored MemoryType.EPISODIC memory: I like flying on Delta when possible'

In [ ]:

'''Retrieve Memories Tool

This tool allows us to search through our stored memories using semantic similarity and filtering.

This tool is particularly useful when you want to:

    Find relevant past experiences or preferences
    Filter memories by type (episodic or semantic)
    Get user-specific information
    Limit the number of results to keep responses focused

The tool works by:

    Taking a query string and searching for semantically similar memories
    Filtering results based on memory type
    Applying a similarity threshold to ensure relevance
    Formatting the results in a clear, readable way
'''
@tool
def retrieve_memories_tool(
    query: str,
    memory_type: List[MemoryType],
    limit: int = 5,
    config: Optional[RunnableConfig] = None,
) -> str:
    """
    Retrieve long-term memories relevant to the query.

    Use this tool to access previously stored information about user
    preferences, experiences, or general knowledge.
    """
    config = config or RunnableConfig()
    user_id = config.get("user_id", SYSTEM_USER_ID)

    try:
        # Get long-term memories
        stored_memories = retrieve_memories(
            query=query,
            memory_type=memory_type,
            user_id=user_id,
            limit=limit,
            distance_threshold=0.3,
        )

        # Format the response
        response = []

        if stored_memories:
            response.append("Long-term memories:")
            for memory in stored_memories:
                response.append(f"- [{memory.memory_type}] {memory.content}")

        return "\n".join(response) if response else "No relevant memories found."

    except Exception as e:
        return f"Error retrieving memories: {str(e)}"

In [38]:
retrieve_memories_tool.invoke({"query": "Airline preferences", "memory_type": ["episodic"]})

'Long-term memories:\n- [MemoryType.EPISODIC] I like flying on Delta when possible'